Import Libraries

In [34]:
import kagglehub as kh
import pandas as pd
import os
import matplotlib.pyplot as plt
import numpy as np
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import PolynomialFeatures
from sklearn.preprocessing import StandardScaler

Download latest version of Dataset

In [2]:
path = kh.dataset_download("amruthayenikonda/dirty-dataset-to-practice-data-cleaning")

print("Path to dataset files:", path)
os.listdir(path)

100%|██████████| 1.21k/1.21k [00:00<00:00, 485kB/s]

Extracting files...
Path to dataset files: /root/.cache/kagglehub/datasets/amruthayenikonda/dirty-dataset-to-practice-data-cleaning/versions/1


['my_file (1).csv']

In [3]:
df = pd.read_csv(path + '/my_file (1).csv')
df.head()

,Rank,Peak,All Time Peak,Actual gross,Adjusted gross (in 2022 dollars),Artist,Tour title,Year(s),Shows,Average gross,Ref.
0,1,1,2,"$780,000,000","$780,000,000",Taylor Swift,The Eras Tour †,2023–2024,56,"$13,928,571",[1]
1,2,1,7[2],"$579,800,000","$579,800,000",Beyoncé,Renaissance World Tour,2023,56,"$10,353,571",[3]
2,3,1[4],2[5],"$411,000,000","$560,622,615",Madonna,Sticky & Sweet Tour ‡[4][a],2008–2009,85,"$4,835,294",[6]
3,4,2[7],10[7],"$397,300,000","$454,751,555",Pink,Beautiful Trauma World Tour,2018–2019,156,"$2,546,795",[7]
4,5,2[4],NaN,"$345,675,146","$402,844,849",Taylor Swift,Reputation Stadium Tour,2018,53,"$6,522,173",[8]


In [4]:
analysis = [
    {
        "column": "Rank",
        "dtype": "int64",
        "issues": "Rank is derived from actual gross and not a numeric scale itself, so if our target or prediction is actual gross, Rank will cause a data leakage and over predict",
        "steps": "Drop Column"
    },
    {
        "column": "Peak",
        "dtype": "object",
        "issues": "it has more than 50% missing data and its not relevant for our analysis'",
        "steps": "Drop Column"
    },
    {
        "column": "All Time Peak",
        "dtype": "object",
        "issues": "it has more than 70% missing data and its not relevant for our analysis'",
        "steps": "Drop Column"
    },
    {
        "column": "Actual gross",
        "dtype": "Float",
        "issues": "this column has some special characters",
        "steps": "removing $, ',', [b] and [e]"

    },
    {
        "column": "Adjusted gross (in 2022 dollars)",
        "dtype": "Float",
        "issues": "this column has some special characters",
        "steps": "removing $, ','"

    },
    {
        "column": "Artist",
        "dtype": "Object",
        "issues": "this column is clean",
        "steps": "Nothing"

    },
    {
        "column": "Tour title",
        "dtype": "Object",
        "issues": "this column has some special characters",
        "steps": "removing special characters"

    },
    {
        "column": "Tour title",
        "dtype": "Object",
        "issues": "this column has some special characters",
        "steps": "removing special characters"

    },
    {
        "column": "Year(s)",
        "dtype": "Object",
        "issues": "this column has both the start year and end year",
        "steps": "we created a new column called age, by subtrating the end year from the start year"

    },
    {
        "column": "Shows",
        "dtype": "Int",
        "issues": "this column is clean",
        "steps": "Nothing"

    },
    {
        "column": "Average Gross",
        "dtype": "Float",
        "issues": "this column has some special characters",
        "steps": "removing $, ','"
    },
    {
        "column": "Ref.",
        "dtype": "object",
        "issues": "we couldnt find the relevance of this column",
        "steps": "Drop Column to reduce redundant data"
    }
]

table = pd.DataFrame(analysis)
table

,column,dtype,issues,steps
0,Rank,int64,Rank is derived from actual gross and not a nu...,Drop Column
1,Peak,object,it has more than 50% missing data and its not ...,Drop Column
2,All Time Peak,object,it has more than 70% missing data and its not ...,Drop Column
3,Actual gross,Float,this column has some special characters,"removing $, ',', [b] and [e]"
4,Adjusted gross (in 2022 dollars),Float,this column has some special characters,"removing $, ','"
5,Artist,Object,this column is clean,Nothing
6,Tour title,Object,this column has some special characters,removing special characters
7,Tour title,Object,this column has some special characters,removing special characters
8,Year(s),Object,this column has both the start year and end year,"we created a new column called age, by subtrat..."
9,Shows,Int,this column is clean,Nothing


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 11 columns):
 #   Column                            Non-Null Count  Dtype 
---  ------                            --------------  ----- 
 0   Rank                              20 non-null     int64 
 1   Peak                              9 non-null      object
 2   All Time Peak                     6 non-null      object
 3   Actual gross                      20 non-null     object
 4   Adjusted gross (in 2022 dollars)  20 non-null     object
 5   Artist                            20 non-null     object
 6   Tour title                        20 non-null     object
 7   Year(s)                           20 non-null     object
 8   Shows                             20 non-null     int64 
 9   Average gross                     20 non-null     object
 10  Ref.                              20 non-null     object
dtypes: int64(2), object(9)
memory usage: 1.8+ KB


## Checking for missing values

it was observed that 70% of data is missing from the **All Time Peak** column and 55% missing from the **Peak** column. this has made reduced the relevance of this columns, therefore this columns will be dropped from the dataset.

Another Irrelevant column to be dropped is **Ref.** it has no relevence for our analysis yet

In [6]:
df.isna().sum()/ df.shape[0]


,0
Rank,0.00
Peak,0.55
All Time Peak,0.70
Actual gross,0.00
Adjusted gross (in 2022 dollars),0.00
Artist,0.00
Tour title,0.00
Year(s),0.00
Shows,0.00
Average gross,0.00


In [7]:
df.drop(columns=['Peak', 'All Time Peak', 'Ref.'], inplace = True)

In [8]:
df.head()

,Rank,Actual gross,Adjusted gross (in 2022 dollars),Artist,Tour title,Year(s),Shows,Average gross
0,1,"$780,000,000","$780,000,000",Taylor Swift,The Eras Tour †,2023–2024,56,"$13,928,571"
1,2,"$579,800,000","$579,800,000",Beyoncé,Renaissance World Tour,2023,56,"$10,353,571"
2,3,"$411,000,000","$560,622,615",Madonna,Sticky & Sweet Tour ‡[4][a],2008–2009,85,"$4,835,294"
3,4,"$397,300,000","$454,751,555",Pink,Beautiful Trauma World Tour,2018–2019,156,"$2,546,795"
4,5,"$345,675,146","$402,844,849",Taylor Swift,Reputation Stadium Tour,2018,53,"$6,522,173"


Check for duplicated rows

In [9]:
duplicate = df.duplicated().sum()
print(duplicate)

0


We will now check our columns for inconsistent data

In [10]:
for a in df.columns:
  print(f'{a}:\n {df[a].unique()}\n')

Rank:
 [ 1  2  3  4  5  6  7  9 10 11 12 13 14 15 16 17 18 19 20]

Actual gross:
 ['$780,000,000' '$579,800,000' '$411,000,000' '$397,300,000'
 '$345,675,146' '$305,158,363' '$280,000,000' '$257,600,000'
 '$256,084,556' '$250,400,000' '$229,100,000[b]' '$227,400,000'
 '$204,000,000' '$200,000,000' '$194,000,000' '$184,000,000'
 '$170,000,000' '$169,800,000' '$167,700,000[e]' '$150,000,000']

Adjusted gross (in 2022 dollars):
 ['$780,000,000' '$579,800,000' '$560,622,615' '$454,751,555'
 '$402,844,849' '$388,978,496' '$381,932,682' '$257,600,000'
 '$312,258,401' '$309,141,878' '$283,202,896' '$295,301,479'
 '$251,856,802' '$299,676,265' '$281,617,035' '$227,452,347'
 '$213,568,571' '$207,046,755' '$204,486,106' '$185,423,109']

Artist:
 ['Taylor Swift' 'Beyoncé' 'Madonna' 'Pink' 'Celine Dion' 'Lady Gaga'
 'Katy Perry' 'Cher' 'Adele']

Tour title:
 ['The Eras Tour †' 'Renaissance World Tour' 'Sticky & Sweet Tour ‡[4][a]'
 'Beautiful Trauma World Tour' 'Reputation Stadium Tour' 'The MDNA 

We were unable to do some data cleaning on columns because of hidden characters on column headers

we print out the column heads and use replace function to remove hidden characters

In [11]:
for col in df.columns:
    print(repr(col))

'Rank'
'Actual\xa0gross'
'Adjusted\xa0gross (in 2022 dollars)'
'Artist'
'Tour title'
'Year(s)'
'Shows'
'Average gross'


In [12]:
df.columns = df.columns.str.replace('\xa0', ' ')
for col in df.columns:
    print(repr(col))

'Rank'
'Actual gross'
'Adjusted gross (in 2022 dollars)'
'Artist'
'Tour title'
'Year(s)'
'Shows'
'Average gross'


Now we create a function to clean the amount columns

removing $, ',', [b] and [e]

In [13]:
def clean_amount(col):
   return (
        col.astype(str)
           .str.replace('$', '', regex=False)
           .str.replace(',', '', regex=False)
           .str.replace('[b]', '', regex=False)
           .str.replace('[e]', '', regex=False)
           .str.strip()
           .replace('', np.nan)
           .astype(float)
    )

we apply the function to the various column to clean the data

In [14]:
df['Actual gross'] = clean_amount(df['Actual gross'])
df['Adjusted gross (in 2022 dollars)'] = clean_amount(df['Adjusted gross (in 2022 dollars)'])
df['Average gross'] = clean_amount(df['Average gross'])

we check the data again to see if the columns are clean

In [15]:
for a in df.columns:
  print(f'{a}:\n {df[a].unique()}\n')

Rank:
 [ 1  2  3  4  5  6  7  9 10 11 12 13 14 15 16 17 18 19 20]

Actual gross:
 [7.80000000e+08 5.79800000e+08 4.11000000e+08 3.97300000e+08
 3.45675146e+08 3.05158363e+08 2.80000000e+08 2.57600000e+08
 2.56084556e+08 2.50400000e+08 2.29100000e+08 2.27400000e+08
 2.04000000e+08 2.00000000e+08 1.94000000e+08 1.84000000e+08
 1.70000000e+08 1.69800000e+08 1.67700000e+08 1.50000000e+08]

Adjusted gross (in 2022 dollars):
 [7.80000000e+08 5.79800000e+08 5.60622615e+08 4.54751555e+08
 4.02844849e+08 3.88978496e+08 3.81932682e+08 2.57600000e+08
 3.12258401e+08 3.09141878e+08 2.83202896e+08 2.95301479e+08
 2.51856802e+08 2.99676265e+08 2.81617035e+08 2.27452347e+08
 2.13568571e+08 2.07046755e+08 2.04486106e+08 1.85423109e+08]

Artist:
 ['Taylor Swift' 'Beyoncé' 'Madonna' 'Pink' 'Celine Dion' 'Lady Gaga'
 'Katy Perry' 'Cher' 'Adele']

Tour title:
 ['The Eras Tour †' 'Renaissance World Tour' 'Sticky & Sweet Tour ‡[4][a]'
 'Beautiful Trauma World Tour' 'Reputation Stadium Tour' 'The MDNA Tour'


In [16]:
df.head()

,Rank,Actual gross,Adjusted gross (in 2022 dollars),Artist,Tour title,Year(s),Shows,Average gross
0,1,780000000.0,780000000.0,Taylor Swift,The Eras Tour †,2023–2024,56,13928571.0
1,2,579800000.0,579800000.0,Beyoncé,Renaissance World Tour,2023,56,10353571.0
2,3,411000000.0,560622615.0,Madonna,Sticky & Sweet Tour ‡[4][a],2008–2009,85,4835294.0
3,4,397300000.0,454751555.0,Pink,Beautiful Trauma World Tour,2018–2019,156,2546795.0
4,5,345675146.0,402844849.0,Taylor Swift,Reputation Stadium Tour,2018,53,6522173.0


Our Year(s) column has a start and end year in it
we need to get an age colunm (diff start and end year)

so we first change the column type to str, then we normalize the hyphen, split the column and assigned them to the different planned columns

In [17]:
df['Year(s)'] = df['Year(s)'].astype(str)

 # Normalize different dash types to standard hyphen
df['Year(s)'] = df['Year(s)'].str.replace('–', '-', regex=False)

# Split the year column
year_split = df['Year(s)'].str.split('-', expand=True)
# Create new columns
df['start_year'] = year_split[0].astype(int)
df['end_year'] = year_split[1].fillna(year_split[0]).astype(int)
df['Age'] = df['end_year'] - df['start_year']
df.drop(columns=['Year(s)', 'start_year', 'end_year'], inplace = True)

In [18]:
df.head()

,Rank,Actual gross,Adjusted gross (in 2022 dollars),Artist,Tour title,Shows,Average gross,Age
0,1,780000000.0,780000000.0,Taylor Swift,The Eras Tour †,56,13928571.0,1
1,2,579800000.0,579800000.0,Beyoncé,Renaissance World Tour,56,10353571.0,0
2,3,411000000.0,560622615.0,Madonna,Sticky & Sweet Tour ‡[4][a],85,4835294.0,1
3,4,397300000.0,454751555.0,Pink,Beautiful Trauma World Tour,156,2546795.0,1
4,5,345675146.0,402844849.0,Taylor Swift,Reputation Stadium Tour,53,6522173.0,0


In [19]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 8 columns):
 #   Column                            Non-Null Count  Dtype  
---  ------                            --------------  -----  
 0   Rank                              20 non-null     int64  
 1   Actual gross                      20 non-null     float64
 2   Adjusted gross (in 2022 dollars)  20 non-null     float64
 3   Artist                            20 non-null     object 
 4   Tour title                        20 non-null     object 
 5   Shows                             20 non-null     int64  
 6   Average gross                     20 non-null     float64
 7   Age                               20 non-null     int64  
dtypes: float64(3), int64(3), object(2)
memory usage: 1.4+ KB


Cleaning Tour Title

here we will replace some special character on the tour title column with nothing

In [20]:
import re

df['Tour title'] = (
    df['Tour title']
    #.str.replace(r'[\[\]†‡\*\d\w+]', '', regex=True)
    .str.replace('‡[4][a]', '', regex=False)
    .str.replace('†', '', regex=False)
    .str.replace('‡[21][a]', '', regex=False)
    .str.replace('*', '', regex=False)
    .str.strip()
    .str.lower()
)

What do we do with Rank? Rank is derived from actual gross and not a numeric scale itself, so if our target or prediction is actual gross, Rank will cause a data leakage and over predict so its better we drop rank

In [21]:
df.drop(columns=['Rank'], inplace = True)

##**Feature Engineering**

Firstly we are focusing on Artist, it is a norminal categorical variable, and its values is less than 50 so we can use a One Hot Encoding to change the features for Machine learning

after creating the encoded dataframe, we joined to the main dataframe

In [22]:
ohe = OneHotEncoder(sparse_output = False, handle_unknown= "ignore")
encoded = ohe.fit_transform(df[['Artist']])
encoded_df = pd.DataFrame(encoded, columns = ohe.get_feature_names_out(['Artist']))
df = pd.concat([df, encoded_df], axis=1)
df.drop(columns=['Artist'], inplace = True)
df.head()

,Actual gross,Adjusted gross (in 2022 dollars),Tour title,Shows,Average gross,Age,Artist_Adele,Artist_Beyoncé,Artist_Celine Dion,Artist_Cher,Artist_Katy Perry,Artist_Lady Gaga,Artist_Madonna,Artist_Pink,Artist_Taylor Swift
0,780000000.0,780000000.0,the eras tour,56,13928571.0,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
1,579800000.0,579800000.0,renaissance world tour,56,10353571.0,0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,411000000.0,560622615.0,sticky & sweet tour,85,4835294.0,1,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
3,397300000.0,454751555.0,beautiful trauma world tour,156,2546795.0,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
4,345675146.0,402844849.0,reputation stadium tour,53,6522173.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0


In [23]:
df = pd.concat([df, encoded_df], axis=1)
df.head()

,Actual gross,Adjusted gross (in 2022 dollars),Tour title,Shows,Average gross,Age,Artist_Adele,Artist_Beyoncé,Artist_Celine Dion,Artist_Cher,...,Artist_Taylor Swift,Artist_Adele,Artist_Beyoncé,Artist_Celine Dion,Artist_Cher,Artist_Katy Perry,Artist_Lady Gaga,Artist_Madonna,Artist_Pink,Artist_Taylor Swift
0,780000000.0,780000000.0,the eras tour,56,13928571.0,1,0.0,0.0,0.0,0.0,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
1,579800000.0,579800000.0,renaissance world tour,56,10353571.0,0,0.0,1.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,411000000.0,560622615.0,sticky & sweet tour,85,4835294.0,1,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
3,397300000.0,454751555.0,beautiful trauma world tour,156,2546795.0,1,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
4,345675146.0,402844849.0,reputation stadium tour,53,6522173.0,0,0.0,0.0,0.0,0.0,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0


we created binary columns with keywords from Tour Title

In [24]:
title = df['Tour title'].str.lower()

keywords = {
    'world_tour': 'world',
    'farewell': 'farewell',
    'live': 'live',
    'stadium': 'stadium',
    'tour': 'tour'
}

for col, word in keywords.items():
    df[col] = title.str.contains(word).astype(int)

df.drop(columns=['Tour title'], inplace = True)

In [25]:
df.head()

,Actual gross,Adjusted gross (in 2022 dollars),Shows,Average gross,Age,Artist_Adele,Artist_Beyoncé,Artist_Celine Dion,Artist_Cher,Artist_Katy Perry,...,Artist_Katy Perry,Artist_Lady Gaga,Artist_Madonna,Artist_Pink,Artist_Taylor Swift,world_tour,farewell,live,stadium,tour
0,780000000.0,780000000.0,56,13928571.0,1,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0,0,0,0,1
1,579800000.0,579800000.0,56,10353571.0,0,0.0,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1,0,0,0,1
2,411000000.0,560622615.0,85,4835294.0,1,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0,0,0,0,1
3,397300000.0,454751555.0,156,2546795.0,1,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,1,0,0,0,1
4,345675146.0,402844849.0,53,6522173.0,0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0,0,0,1,1


##**Creation of New Features**

we created mathematical columns usings logs and divisions

In [26]:
df['Log_actual_gross'] = np.log(df['Actual gross'])
df['Log_avg_gross'] = np.log(df['Average gross'])
df['Adjusted_gross'] = np.log(df['Adjusted gross (in 2022 dollars)'])
df['Gross_per_show'] = df['Actual gross'] / df['Shows']
df['Adjusted_gross_per_show'] = df['Adjusted gross (in 2022 dollars)'] / df['Shows']

In [27]:
df.head()

,Actual gross,Adjusted gross (in 2022 dollars),Shows,Average gross,Age,Artist_Adele,Artist_Beyoncé,Artist_Celine Dion,Artist_Cher,Artist_Katy Perry,...,world_tour,farewell,live,stadium,tour,Log_actual_gross,Log_avg_gross,Adjusted_gross,Gross_per_show,Adjusted_gross_per_show
0,780000000.0,780000000.0,56,13928571.0,1,0.0,0.0,0.0,0.0,0.0,...,0,0,0,0,1,20.474804,16.449453,20.474804,1.392857e+07,1.392857e+07
1,579800000.0,579800000.0,56,10353571.0,0,0.0,1.0,0.0,0.0,0.0,...,1,0,0,0,1,20.178194,16.152842,20.178194,1.035357e+07,1.035357e+07
2,411000000.0,560622615.0,85,4835294.0,1,0.0,0.0,0.0,0.0,0.0,...,0,0,0,0,1,19.834104,15.391452,20.144559,4.835294e+06,6.595560e+06
3,397300000.0,454751555.0,156,2546795.0,1,0.0,0.0,0.0,0.0,0.0,...,1,0,0,0,1,19.800202,14.750346,19.935262,2.546795e+06,2.915074e+06
4,345675146.0,402844849.0,53,6522173.0,0,0.0,0.0,0.0,0.0,0.0,...,0,0,0,1,1,19.661010,15.690718,19.814062,6.522173e+06,7.600846e+06


##Creating Polynomial Features

we create more features using polinomial and add to the dataset

In [28]:
poly_cols = [
    'Shows',
    'Age',
    'Log_avg_gross',
    'Log_actual_gross' ]
poly = PolynomialFeatures(degree=2, include_bias=False)
x_features = poly.fit_transform(df[poly_cols])
poly_feature_names = poly.get_feature_names_out(poly_cols)

df_poly = pd.DataFrame(
    x_features,
    columns=poly_feature_names,
    index=df.index
)
df_poly.head()

,Shows,Age,Log_avg_gross,Log_actual_gross,Shows^2,Shows Age,Shows Log_avg_gross,Shows Log_actual_gross,Age^2,Age Log_avg_gross,Age Log_actual_gross,Log_avg_gross^2,Log_avg_gross Log_actual_gross,Log_actual_gross^2
0,56.0,1.0,16.449453,20.474804,3136.0,56.0,921.169354,1146.589051,1.0,16.449453,20.474804,270.584496,336.799329,419.217618
1,56.0,0.0,16.152842,20.178194,3136.0,0.0,904.559154,1129.978851,0.0,0.000000,0.000000,260.914306,325.935177,407.159504
2,85.0,1.0,15.391452,19.834104,7225.0,85.0,1308.273462,1685.898821,1.0,15.391452,19.834104,236.896810,305.275666,393.391672
3,156.0,1.0,14.750346,19.800202,24336.0,156.0,2301.054017,3088.831546,1.0,14.750346,19.800202,217.572715,292.059839,392.048008
4,53.0,0.0,15.690718,19.661010,2809.0,0.0,831.608063,1042.033530,0.0,0.000000,0.000000,246.198636,308.495367,386.555315


In [29]:
df_final = pd.concat(
    [df, df_poly],
    axis=1
)


We are Printing our final dataset

In [30]:
df_final

,Actual gross,Adjusted gross (in 2022 dollars),Shows,Average gross,Age,Artist_Adele,Artist_Beyoncé,Artist_Celine Dion,Artist_Cher,Artist_Katy Perry,...,Shows^2,Shows Age,Shows Log_avg_gross,Shows Log_actual_gross,Age^2,Age Log_avg_gross,Age Log_actual_gross,Log_avg_gross^2,Log_avg_gross Log_actual_gross,Log_actual_gross^2
0,780000000.0,780000000.0,56,13928571.0,1,0.0,0.0,0.0,0.0,0.0,...,3136.0,56.0,921.169354,1146.589051,1.0,16.449453,20.474804,270.584496,336.799329,419.217618
1,579800000.0,579800000.0,56,10353571.0,0,0.0,1.0,0.0,0.0,0.0,...,3136.0,0.0,904.559154,1129.978851,0.0,0.000000,0.000000,260.914306,325.935177,407.159504
2,411000000.0,560622615.0,85,4835294.0,1,0.0,0.0,0.0,0.0,0.0,...,7225.0,85.0,1308.273462,1685.898821,1.0,15.391452,19.834104,236.896810,305.275666,393.391672
3,397300000.0,454751555.0,156,2546795.0,1,0.0,0.0,0.0,0.0,0.0,...,24336.0,156.0,2301.054017,3088.831546,1.0,14.750346,19.800202,217.572715,292.059839,392.048008
4,345675146.0,402844849.0,53,6522173.0,0,0.0,0.0,0.0,0.0,0.0,...,2809.0,0.0,831.608063,1042.033530,0.0,0.000000,0.000000,246.198636,308.495367,386.555315
5,305158363.0,388978496.0,88,3467709.0,0,0.0,0.0,0.0,0.0,0.0,...,7744.0,0.0,1325.192414,1719.198045,0.0,0.000000,0.000000,226.773623,294.197857,381.668636
6,280000000.0,381932682.0,131,2137405.0,1,0.0,0.0,1.0,0.0,0.0,...,17161.0,131.0,1909.338498,2547.989321,1.0,14.575103,19.450300,212.433628,283.490129,378.314176
7,257600000.0,257600000.0,41,6282927.0,1,0.0,0.0,0.0,0.0,0.0,...,1681.0,41.0,641.787207,794.043661,1.0,15.653347,19.366919,245.027257,303.157087,375.077534
8,256084556.0,312258401.0,49,5226215.0,0,0.0,1.0,0.0,0.0,0.0,...,2401.0,0.0,757.990695,948.689894,0.0,0.000000,0.000000,239.296083,299.499422,374.849027
9,250400000.0,309141878.0,85,2945882.0,0,0.0,0.0,0.0,0.0,0.0,...,7225.0,0.0,1266.153100,1643.778467,0.0,0.000000,0.000000,221.888398,288.065772,373.980297


In [31]:
for col in df_final.columns:
    print(repr(col))

'Actual gross'
'Adjusted gross (in 2022 dollars)'
'Shows'
'Average gross'
'Age'
'Artist_Adele'
'Artist_Beyoncé'
'Artist_Celine Dion'
'Artist_Cher'
'Artist_Katy Perry'
'Artist_Lady Gaga'
'Artist_Madonna'
'Artist_Pink'
'Artist_Taylor Swift'
'Artist_Adele'
'Artist_Beyoncé'
'Artist_Celine Dion'
'Artist_Cher'
'Artist_Katy Perry'
'Artist_Lady Gaga'
'Artist_Madonna'
'Artist_Pink'
'Artist_Taylor Swift'
'world_tour'
'farewell'
'live'
'stadium'
'tour'
'Log_actual_gross'
'Log_avg_gross'
'Adjusted_gross'
'Gross_per_show'
'Adjusted_gross_per_show'
'Shows'
'Age'
'Log_avg_gross'
'Log_actual_gross'
'Shows^2'
'Shows Age'
'Shows Log_avg_gross'
'Shows Log_actual_gross'
'Age^2'
'Age Log_avg_gross'
'Age Log_actual_gross'
'Log_avg_gross^2'
'Log_avg_gross Log_actual_gross'
'Log_actual_gross^2'


In [36]:
df_final[['Actual gross', 'Adjusted gross (in 2022 dollars)', 'Average gross','Log_actual_gross', 'Age Log_actual_gross', 'Age', 'Log_actual_gross^2','Shows']].head()

,Actual gross,Adjusted gross (in 2022 dollars),Average gross,Log_actual_gross,Log_actual_gross,Age Log_actual_gross,Age,Age,Log_actual_gross^2,Shows,Shows
0,780000000.0,780000000.0,13928571.0,20.474804,20.474804,20.474804,1,1.0,419.217618,56,56.0
1,579800000.0,579800000.0,10353571.0,20.178194,20.178194,0.000000,0,0.0,407.159504,56,56.0
2,411000000.0,560622615.0,4835294.0,19.834104,19.834104,19.834104,1,1.0,393.391672,85,85.0
3,397300000.0,454751555.0,2546795.0,19.800202,19.800202,19.800202,1,1.0,392.048008,156,156.0
4,345675146.0,402844849.0,6522173.0,19.661010,19.661010,0.000000,0,0.0,386.555315,53,53.0


In [35]:
col_with_large_value = ['Actual gross', 'Adjusted gross (in 2022 dollars)', 'Average gross']

In [37]:
scaler = StandardScaler()

scaled = scaler.fit_transform(df[col_with_large_value])
scaled

array([[ 3.22930225,  2.9542038 ,  3.08458104],
       [ 1.91539615,  1.59808848,  2.00367755],
       [ 0.80756723,  1.46818466,  0.33522305],
       [ 0.71765457,  0.75103497, -0.35670606],
       [ 0.37884233,  0.39942918,  0.84525197],
       [ 0.112932  ,  0.30550124, -0.07826713],
       [-0.05218152,  0.25777429, -0.48048538],
       [-0.19919199, -0.58443078,  0.77291579],
       [-0.2091378 , -0.21418555,  0.45341826],
       [-0.24644536, -0.23529626, -0.23604185],
       [-0.38623657, -0.41100181, -0.60196956],
       [-0.39739361, -0.3290484 , -0.78863449],
       [-0.55096705, -0.62333407, -0.71825754],
       [-0.57721892, -0.29941446, -0.94066908],
       [-0.61659673, -0.42174412, -0.14913086],
       [-0.6822264 , -0.78864504, -0.73495274],
       [-0.77410795, -0.882691  , -0.6022453 ],
       [-0.77542054, -0.9268685 , -0.500644  ],
       [-0.78920277, -0.94421383, -0.70768827],
       [-0.9053673 , -1.07334281, -0.59937539]])

In [38]:
scaled_df = pd.DataFrame(
    scaled,
    columns=df[col_with_large_value].columns,
    index=df[col_with_large_value].index
)
scaled_df.head()

,Actual gross,Adjusted gross (in 2022 dollars),Average gross
0,3.229302,2.954204,3.084581
1,1.915396,1.598088,2.003678
2,0.807567,1.468185,0.335223
3,0.717655,0.751035,-0.356706
4,0.378842,0.399429,0.845252


In [39]:
df_final_scaled = pd.concat(
    [scaled_df, df_final],
    axis=1
)

In [40]:
df_final_scaled.head()

,Actual gross,Adjusted gross (in 2022 dollars),Average gross,Actual gross,Adjusted gross (in 2022 dollars),Shows,Average gross,Age,Artist_Adele,Artist_Beyoncé,...,Shows^2,Shows Age,Shows Log_avg_gross,Shows Log_actual_gross,Age^2,Age Log_avg_gross,Age Log_actual_gross,Log_avg_gross^2,Log_avg_gross Log_actual_gross,Log_actual_gross^2
0,3.229302,2.954204,3.084581,780000000.0,780000000.0,56,13928571.0,1,0.0,0.0,...,3136.0,56.0,921.169354,1146.589051,1.0,16.449453,20.474804,270.584496,336.799329,419.217618
1,1.915396,1.598088,2.003678,579800000.0,579800000.0,56,10353571.0,0,0.0,1.0,...,3136.0,0.0,904.559154,1129.978851,0.0,0.000000,0.000000,260.914306,325.935177,407.159504
2,0.807567,1.468185,0.335223,411000000.0,560622615.0,85,4835294.0,1,0.0,0.0,...,7225.0,85.0,1308.273462,1685.898821,1.0,15.391452,19.834104,236.896810,305.275666,393.391672
3,0.717655,0.751035,-0.356706,397300000.0,454751555.0,156,2546795.0,1,0.0,0.0,...,24336.0,156.0,2301.054017,3088.831546,1.0,14.750346,19.800202,217.572715,292.059839,392.048008
4,0.378842,0.399429,0.845252,345675146.0,402844849.0,53,6522173.0,0,0.0,0.0,...,2809.0,0.0,831.608063,1042.033530,0.0,0.000000,0.000000,246.198636,308.495367,386.555315


In [41]:
df_final_scaled = df_final_scaled.loc[:, ~df_final_scaled.columns.duplicated()]
df_final_scaled

,Actual gross,Adjusted gross (in 2022 dollars),Average gross,Shows,Age,Artist_Adele,Artist_Beyoncé,Artist_Celine Dion,Artist_Cher,Artist_Katy Perry,...,Shows^2,Shows Age,Shows Log_avg_gross,Shows Log_actual_gross,Age^2,Age Log_avg_gross,Age Log_actual_gross,Log_avg_gross^2,Log_avg_gross Log_actual_gross,Log_actual_gross^2
0,3.229302,2.954204,3.084581,56,1,0.0,0.0,0.0,0.0,0.0,...,3136.0,56.0,921.169354,1146.589051,1.0,16.449453,20.474804,270.584496,336.799329,419.217618
1,1.915396,1.598088,2.003678,56,0,0.0,1.0,0.0,0.0,0.0,...,3136.0,0.0,904.559154,1129.978851,0.0,0.000000,0.000000,260.914306,325.935177,407.159504
2,0.807567,1.468185,0.335223,85,1,0.0,0.0,0.0,0.0,0.0,...,7225.0,85.0,1308.273462,1685.898821,1.0,15.391452,19.834104,236.896810,305.275666,393.391672
3,0.717655,0.751035,-0.356706,156,1,0.0,0.0,0.0,0.0,0.0,...,24336.0,156.0,2301.054017,3088.831546,1.0,14.750346,19.800202,217.572715,292.059839,392.048008
4,0.378842,0.399429,0.845252,53,0,0.0,0.0,0.0,0.0,0.0,...,2809.0,0.0,831.608063,1042.033530,0.0,0.000000,0.000000,246.198636,308.495367,386.555315
5,0.112932,0.305501,-0.078267,88,0,0.0,0.0,0.0,0.0,0.0,...,7744.0,0.0,1325.192414,1719.198045,0.0,0.000000,0.000000,226.773623,294.197857,381.668636
6,-0.052182,0.257774,-0.480485,131,1,0.0,0.0,1.0,0.0,0.0,...,17161.0,131.0,1909.338498,2547.989321,1.0,14.575103,19.450300,212.433628,283.490129,378.314176
7,-0.199192,-0.584431,0.772916,41,1,0.0,0.0,0.0,0.0,0.0,...,1681.0,41.0,641.787207,794.043661,1.0,15.653347,19.366919,245.027257,303.157087,375.077534
8,-0.209138,-0.214186,0.453418,49,0,0.0,1.0,0.0,0.0,0.0,...,2401.0,0.0,757.990695,948.689894,0.0,0.000000,0.000000,239.296083,299.499422,374.849027
9,-0.246445,-0.235296,-0.236042,85,0,0.0,0.0,0.0,0.0,0.0,...,7225.0,0.0,1266.153100,1643.778467,0.0,0.000000,0.000000,221.888398,288.065772,373.980297
